# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guided workflow for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if it is not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load the Croissant metadata and review the top-level dataset description. This includes core identifiers, citations, and a high-level summary of what the dataset contains.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show dataset title and description
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview

Enumerate available record sets and their details. Each record set, field, and column is identified by its `@id`. This is crucial for referencing data elements throughout the workflow.

**Notes:** The FAIR\^2 dataset typically consists of tables (record sets). We will show all available record sets and fields. If the dataset has only a single main table, that will be used.

In [ ]:
# List all available record sets and their field ids for reference
print("Record sets present in the dataset:")
for record_set in metadata.record_sets:
    print(f"  @id: {record_set.id}")
    print(f"    name: {record_set.name}")
    if hasattr(record_set, 'description') and record_set.description:
        print(f"    description: {record_set.description}")
    # List fields (columns) in the record set
    print("    fields (columns):")
    for field in record_set.fields:
        print(f"      @id: {field.id} | name: {field.name} | type: {field.data_type}")

## 3. Data Extraction

Now, load data from a specific record set (table) into a DataFrame for further analysis. Use the record set and field `@id`s from the overview above.

> **Tip:** When referencing record sets and fields, always use their `@id` for clarity and reproducibility.

In [ ]:
# --- Identify which record sets to use ---
# We'll collect all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record set @ids:", record_set_ids)

# We'll load all available record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    # each record is a dict by field @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Columns (by @id): {list(df.columns)}\n")

# Choose one main record set for exploration, as an example:
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    raise RuntimeError("No record sets found in the metadata.")

print(f"Using record set for EDA: {main_record_set_id}\n")
# Show a preview
display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We will:
- Select a numeric field for analysis using its `@id`.
- Filter records based on a threshold for that numeric field.
- Normalize that field.
- Group by a chosen categorical field if available.

Use the previous overview to choose `numeric_field_id` and `group_field_id`. Adjust them as needed if the dataset structure changes.

In [ ]:
# Choose a numeric field (by @id) for demonstration
# If possible, pick a likely numeric column. Adjust based on dataset's actual columns.

df = dataframes[main_record_set_id]
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", numeric_candidates)

if len(numeric_candidates) == 0:
    print("No numeric columns detected, EDA steps will be skipped.")
else:
    # Use the first numeric field
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}\n")
    
    threshold = df[numeric_field].mean()  # as a simple dynamic threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.3f}, count: {len(filtered_df)}\n")

    # Normalize (z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (first rows):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field if any
    # Exclude the numeric we just analyzed
    cat_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
    if cat_candidates:
        group_field = cat_candidates[0]
        print(f"Grouping by: {group_field}\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(grouped_df.head())
    else:
        print("No categorical fields found to group by.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and (if possible) show means grouped by the selected categorical field.

In [ ]:
import matplotlib.pyplot as plt

if len(df) > 0 and len(numeric_candidates) > 0:
    plt.figure(figsize=(6,4))
    df[numeric_field].hist(bins=20)
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    if cat_candidates:
        # Plot mean value by group
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        group_means.plot(kind='barh', figsize=(7,4))
        plt.xlabel(f"Mean {numeric_field}")
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

We loaded and explored the Mass Spectrometry dataset using the `mlcroissant` library, referencing all data structures by their `@id` as best practice. By dynamically selecting and analyzing numeric and categorical fields, you can quickly adapt this notebook for any Croissant-compatible dataset. For additional processing, always consult the dataset's Croissant schema for precise `@id` usage.